# Intermediate Machine Learning 
* Datasets from Housing Prices Competition for Kaggle Learn Users on Kaggle

## Lesson 1: Introduction (Random Forest Tuning)

### 1. The Goal
Real-world machine learning involves tweaking models to find the absolute best predictive performance. In this warm-up, we are going to load the Iowa Housing dataset, define five different Random Forest architectures (using different **hyperparameters**), and write an automated loop to test which one performs the best.

### 2. Loading and Splitting the Data
First, we load our data using Pandas. We separate our target variable (`SalePrice`) from our features. 

To ensure we can accurately test our models, we use `train_test_split` to hide 20% of our training data. This hidden 20% becomes our **Validation Set**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Load the data (Update these paths if you put the CSVs in a different folder)
X_full = pd.read_csv('../data/train.csv', index_col='Id')
X_test_full = pd.read_csv('../data/test.csv', index_col='Id')

# 2. Separate the Target (y) from the Features
y = X_full.SalePrice

# Target - what we are trying to predict
# Features - These are the inputs used to guess the target

# For this warm-up, we are only using 7 specific numeric features
features = ['LotArea', 'YearBuilt', '1stFlrSF', '2ndFlrSF', 'FullBath', 'BedroomAbvGr', 'TotRmsAbvGrd']
X = X_full[features].copy()
X_test = X_test_full[features].copy()

# 3. Break off the validation set from the training data (80% Train / 20% Validate)
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

# Preview the training features
X_train.head()

,LotArea,YearBuilt,1stFlrSF,2ndFlrSF,FullBath,BedroomAbvGr,TotRmsAbvGrd
Id,,,,,,,
619,11694,2007,1828,0,2,3,9
871,6600,1962,894,0,1,2,5
93,13360,1921,964,0,1,2,5
818,13265,2002,1689,0,2,3,7
303,13704,2001,1541,0,2,3,6


### 3. Defining the Competitors (Hyperparameter Tuning)
A Random Forest has several settings (hyperparameters) we can tweak to prevent underfitting or overfitting:
* **`n_estimators`:** The total number of trees in the forest.
* **`criterion`:** The math formula the tree uses to evaluate its splits (e.g., absolute error vs. squared error).
* **`max_depth`:** The maximum number of splits a tree is allowed to make.
* **`min_samples_split`:** The minimum number of houses required in a node before it is allowed to split again.

We will define 5 different models to race against each other.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Define 5 different Random Forest architectures
model_1 = RandomForestRegressor(n_estimators=50, random_state=0)
model_2 = RandomForestRegressor(n_estimators=100, random_state=0)
model_3 = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=0)
model_4 = RandomForestRegressor(n_estimators=200, min_samples_split=20, random_state=0)
model_5 = RandomForestRegressor(n_estimators=100, max_depth=7, random_state=0)

# Group them into a list so we can loop through them
models = [model_1, model_2, model_3, model_4, model_5]

### 4. Evaluating the Models
Instead of writing the training and testing code five separate times, we define a custom Python function `score_model()`. We can pass any model into this function, and it will automatically fit the data, make predictions on the validation set, and return the Mean Absolute Error (MAE).

In [ ]:
from sklearn.metrics import mean_absolute_error

# 1. Define the evaluation function
def score_model(model, X_t=X_train, X_v=X_valid, y_t=y_train, y_v=y_valid):
    model.fit(X_t, y_t)
    preds = model.predict(X_v)
    return mean_absolute_error(y_v, preds)

# 2. Loop through the 5 models and print their scores
print("Evaluating Model Architectures:\n" + "-"*35)
for i in range(0, len(models)):
    mae = score_model(models[i])
    print(f"Model {i+1} MAE: ${mae:,.0f}")


# The best model is the one with the lowest MAE

Evaluating Model Architectures:
-----------------------------------
Model 1 MAE: $24,015
Model 2 MAE: $23,741
Model 3 MAE: $23,529
Model 4 MAE: $23,997
Model 5 MAE: $23,707


### 5. Training the Final Model for Production
After running the evaluation loop, we observe which model achieved the lowest MAE. For this dataset, **Model 3** (100 trees using `absolute_error` criterion) performs the best.

Now that we know the optimal architecture, we define our final model. Because we are ready to predict on the *actual* test data (the unknown houses), we no longer need to hide 20% of our data for validation. We train this final model on **100% of the available training data (`X` and `y`)** to make it as smart as possible.

In [ ]:
# 1. Define the final model using the winning architecture (Model 3)
my_model = RandomForestRegressor(n_estimators=100, criterion='absolute_error', random_state=0)

# 2. Fit the model to ALL of the training data
my_model.fit(X, y)

# 3. Generate predictions for the completely unseen test.csv data
preds_test = my_model.predict(X_test)

# 4. Save the predictions to a CSV file (This is the format Kaggle requires for submission)
output = pd.DataFrame({'Id': X_test.index, 'SalePrice': preds_test})
output.to_csv('submission.csv', index=False)

print("Final predictions successfully saved to submission.csv!")

Final predictions successfully saved to submission.csv!
